# mdparser.ipynb

Tools for extracting data from markdown files.

In [31]:
import re
from icecream import ic
import pandas as pd

In [32]:
def detection_line_to_dict(line: str)->dict:
    """ 
    Returns a dict containing data extracted from a line beginning with "## Detected object" 
    """
    pattern = r'## Detected object (\d+) confidence: ([\d.]+) object_index: (\d+)'

    match = re.search(pattern, line)

    if match:
        # Construct the dictionary with appropriate type casting
        result = {
            'detected_object': int(match.group(1)),
            'confidence': float(match.group(2)),
            'object_index': int(match.group(3))
        }
        
        return result

    return None
        
# # Usage example:
# line = '## Detected object 26 confidence: 0.260 object_index: 1'
# mydict =  detection_line_to_dict(line)
# print(mydict)

In [33]:
def parse_checkbox(line):
    # Regex explanation:
    # -\s* : Match a hyphen followed by optional whitespace
    # \[(.*?)\] : Capture the content inside square brackets
    # \s+       : Match one or more whitespace characters
    # (.*)      : Capture the remaining text as the label
    match = re.search(r'-\s*\[(.*?)\]\s+(.*)', line)
    
    if match:
        status_char = match.group(1).strip().lower()
        label = match.group(2).strip()
        
        # Consider 'x' as True, any other character (like a space) as False
        is_checked = (status_char == 'x')
        
        return {label: is_checked}
    
    return None

# # Example usage:
# line = '- [x] accept'
# result = parse_checkbox(line)
# print(result)  # Output: {'accept': True}


In [35]:
md_path = 'sam3.md'
expected_dict_length = 3+8

with open(md_path) as f:
    lines = f.readlines()

mylist = []    
for line in lines:
    if line.startswith('## Detected object'):
        mydict = detection_line_to_dict(line) 
        
    if line.startswith('- ['):
        mydict = mydict | parse_checkbox(line)
        if len(mydict) == expected_dict_length:
            mylist.append(mydict)
            
df = pd.DataFrame(mylist)
df.to_csv('data/data_for_sam3_post/detection_attributes.csv', index=False)
df
            

,detected_object,confidence,object_index,accept,healthy,damaged,vcuts,dead,crowd,occluded,other_problem
0,2,0.842,16,True,False,True,False,False,False,False,False
1,3,0.771,5,False,True,False,False,False,True,False,False
2,4,0.764,9,True,False,True,False,False,False,False,False
3,5,0.743,3,False,True,False,False,False,False,True,False
4,6,0.730,12,True,False,True,False,False,False,False,False
5,7,0.699,2,True,False,True,False,False,False,False,False
6,8,0.691,20,False,False,False,False,False,False,True,False
7,9,0.678,7,True,False,True,False,False,False,False,False
8,10,0.650,0,True,False,True,False,False,False,False,False
9,11,0.630,24,False,False,False,False,False,True,False,False
